In [1]:
# analise_saguis.py
# -*- coding: utf-8 -*-

# --------------------------------------------------------------------------
# ETAPA: PREPARAÇÃO DO AMBIENTE E CARREGAMENTO DOS DADOS
# --------------------------------------------------------------------------

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# (opcional) para tabelas bonitas, caso use Jupyter
pd.set_option('display.max_columns', 200)

# --- 1. Caminho e leitura ---
CSV_PATH = r"C:/Users/maira/OneDrive/Área de Trabalho/QGIS/extrair-analises-qgis-r/todos-saguis-todas-variaveis-e-grid.csv"
dados_brutos = pd.read_csv(CSV_PATH)

print("Colunas no CSV:")
print(dados_brutos.columns.tolist())
print(dados_brutos.head())
print(dados_brutos.info())

# --------------------------------------------------------------------------
# ETAPA: RENOMEANDO COLUNAS PARA PADRONIZAR (espelha seu rename do R)
# --------------------------------------------------------------------------
dados_padronizados = dados_brutos.rename(columns={
    "id": "id_grid",
    "tipo": "especie",
    "elev_1": "elevacao",
    "temp_1": "temperatura_media_anual",
    "precip_1": "precipitacao_media_anual",
    "mapbio_1": "uso_terra"
})

print("Após padronização:")
print(dados_padronizados.columns.tolist())

# --------------------------------------------------------------------------
# ETAPA: AGREGAÇÃO DOS DADOS (como no seu summarise por id)
#   - Contagens por 'layer' (espécies não híbridas)
#   - Contagem de híbridos por 'tipo' == "H"
# --------------------------------------------------------------------------
def count_equal(series, value):
    # soma de (series == value) com NA-safe
    return np.sum(series.fillna("") == value)

ag = (
    dados_brutos
    .groupby("id", as_index=False)
    .apply(lambda g: pd.Series({
        "C_aurita_n": count_equal(g["layer"], "C. aurita - naohibridos - inat"),
        "C_jacchus_n": count_equal(g["layer"], "C. jacchus - naohibridos - inat"),
        "C_penicillata_n": count_equal(g["layer"], "C. penicillata - naohibridos - inat"),
        "hibridos_n": np.sum(g["tipo"].fillna("") == "H"),
        "total_n": len(g),
        "Bioma": g["Bioma"].iloc[0] if "Bioma" in g and len(g["Bioma"]) else np.nan,
        "elevacao": g["elev_1"].mean(skipna=True),
        "temp_media": g["temp_1"].mean(skipna=True),
        "prec_media": g["precip_1"].mean(skipna=True),
        "uso_terra": g["mapbio_1"].iloc[0] if "mapbio_1" in g and len(g["mapbio_1"]) else np.nan
    }))
    .reset_index(drop=True)
)
dados_agregados = ag.copy()

print("\nDados agregados (head):")
print(dados_agregados.head())
print(dados_agregados.describe(include='all'))

# --------------------------------------------------------------------------
# ETAPA: EDA — médias, variâncias, desvios
# --------------------------------------------------------------------------
contagens_especies = dados_agregados[["C_aurita_n", "C_jacchus_n", "C_penicillata_n"]].copy()

print("\nResumo (médias/medianas, etc.):")
print(contagens_especies.describe())

print("\nVariâncias:")
print(contagens_especies.var())

print("\nDesvios-padrão:")
print(contagens_especies.std())

# --------------------------------------------------------------------------
# ETAPA: TESTE DE CORRELAÇÃO DE PEARSON
#   - n_invasoras = jacchus + penicillata
# --------------------------------------------------------------------------
dados_correlacao = dados_agregados.assign(
    n_invasoras = dados_agregados["C_jacchus_n"] + dados_agregados["C_penicillata_n"]
)

def pearson_report(x, y, nome):
    # remove NAs para o teste
    df = dados_correlacao[[x, y]].dropna()
    r, p = stats.pearsonr(df[x], df[y])
    print(f"\nPearson {nome}: r={r:.3f}, p={p:.3e}, N={len(df)}")

pearson_report("C_aurita_n", "n_invasoras", "C_aurita_n ~ n_invasoras")
pearson_report("hibridos_n", "n_invasoras", "hibridos_n ~ n_invasoras")
pearson_report("hibridos_n", "C_aurita_n", "hibridos_n ~ C_aurita_n")

# --------------------------------------------------------------------------
# MATRIZ DE CORRELAÇÃO ENTRE VARIÁVEIS AMBIENTAIS (+ is_urban / is_vegetation)
# --------------------------------------------------------------------------
dados_para_matriz_ambiental = (
    dados_agregados
    .assign(
        is_urban=lambda d: (d["uso_terra"] == 24).astype(int),
        is_vegetation=lambda d: d["uso_terra"].isin([3,4,12]).astype(int)
    )[["elevacao","temp_media","prec_media","is_urban","is_vegetation"]]
)

matriz_cor = dados_para_matriz_ambiental.corr(numeric_only=True)
print("\nMatriz de correlação (ambiental):")
print(matriz_cor.round(2))

# Correlograma (equivalente ao corrplot)
plt.figure(figsize=(6,5))
sns.heatmap(matriz_cor, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Matriz de Correlação entre Variáveis Ambientais e Climáticas")
plt.tight_layout()
plt.show()

# --------------------------------------------------------------------------
# GLM(s) — aproximação dos seus GLMMs (efeitos fixos: Bioma como dummies)
# --------------------------------------------------------------------------
# 1) Preparar dataframe final
dfm = dados_agregados.copy()
dfm = dfm[dfm["total_n"] > 0].copy()

dfm["n_invasoras"] = dfm["C_jacchus_n"] + dfm["C_penicillata_n"]
dfm["zona_sobreposicao"] = (dfm["C_aurita_n"] > 0) & (dfm["n_invasoras"] > 0)

# uso_terra_agrupado (como no seu case_when)
def map_uso_terra(u):
    if u == 3:
        return "Floresta"
    if u == 24:
        return "Area_Urbanizada"
    if u in [15,19,21,36,49]:
        return "Uso_Antropico_Rural"
    return "Outros_e_Abertos"

dfm["uso_terra_agrupado"] = dfm["uso_terra"].apply(map_uso_terra)
dfm["uso_terra_agrupado"] = pd.Categorical(dfm["uso_terra_agrupado"], 
                                           categories=["Floresta","Area_Urbanizada","Uso_Antropico_Rural","Outros_e_Abertos"])
dfm["Bioma"] = dfm["Bioma"].astype("category")
dfm = dfm.dropna(subset=["elevacao","prec_media","uso_terra_agrupado","Bioma"])

# 2) Modelo "Q4 final" no R: Poisson com link log, (1|Bioma)
#    Aqui: GLM Poisson com Bioma como efeito fixo
form_q4 = "C_aurita_n ~ scale_elev + scale_prec + C(uso_terra_agrupado) + C(Bioma)"
dfm["scale_elev"] = (dfm["elevacao"] - dfm["elevacao"].mean())/dfm["elevacao"].std(ddof=0)
dfm["scale_prec"] = (dfm["prec_media"] - dfm["prec_media"].mean())/dfm["prec_media"].std(ddof=0)

glm_q4 = smf.glm(formula=form_q4, data=dfm, family=sm.families.Poisson()).fit()
print("\n--- Resultados GLM (aprox. Q4) Poisson (efeitos fixos) ---")
print(glm_q4.summary())

# Checagem de (sobre)dispersão simples (phi = deviance/df_resid > ~1 indica sobre-dispersão)
phi_q4 = glm_q4.deviance / glm_q4.df_resid
print(f"\nRazão de dispersão (phi) ~ {phi_q4:.2f}  ( >1 sugere sobre-dispersão )")

# 3) "Q5 final" no R: Binomial (hibridos_n / total_n-hibridos_n) ~ zona_sobreposicao + (1|Bioma)
#    Aqui: GLM Binomial com Bioma como efeito fixo
#    Monta resposta binomial como 'successes' e 'failures' via weights + proporção
dfm = dfm.copy()
dfm["prop_hibridos"] = np.where(dfm["total_n"]>0, dfm["hibridos_n"]/dfm["total_n"], np.nan)
dfm = dfm.dropna(subset=["prop_hibridos"])  # remove linhas sem total_n

form_q5 = "prop_hibridos ~ zona_sobreposicao + C(Bioma) + 0"  # sem intercepto para facilitar leitura dos biomas
glm_q5 = smf.glm(formula=form_q5, data=dfm, 
                 family=sm.families.Binomial(), 
                 freq_weights=dfm["total_n"]).fit()
print("\n--- GLM Binomial (aprox. Q5) com pesos = total_n ---")
print(glm_q5.summary())

# 4) Tabela de contingência (tem_hibrido vs zona_sobreposicao)
dfm["tem_hibrido"] = dfm["hibridos_n"] > 0
tab = pd.crosstab(dfm["tem_hibrido"], dfm["zona_sobreposicao"])
tab.index = ["Células SEM Híbridos","Células COM Híbridos"]
if tab.shape[1] == 1:
    tab.columns = ["Dentro da Zona de Sobreposição"] if tab.columns[0] else ["Fora da Zona de Sobreposição"]
elif tab.shape[1] == 2:
    tab.columns = ["Fora da Zona de Sobreposição","Dentro da Zona de Sobreposição"]
print("\n--- Tabela de Contingência: Híbridos vs. Zona de Sobreposição ---")
print(tab)

# 5) Modelo opcional: proporção de híbridos ~ n_invasoras (binomial)
form_opt = "prop_hibridos ~ scale_ninv + C(Bioma)"
dfm["scale_ninv"] = (dfm["n_invasoras"] - dfm["n_invasoras"].mean())/dfm["n_invasoras"].std(ddof=0) if dfm["n_invasoras"].std(ddof=0) else 0.0
glm_opt = smf.glm(formula=form_opt, data=dfm,
                  family=sm.families.Binomial(),
                  freq_weights=dfm["total_n"]).fit()
print("\n--- GLM Binomial (proporção de híbridos ~ n_invasoras) ---")
print(glm_opt.summary())

# --------------------------------------------------------------------------
# SULDESTE (Mata Atlântica vs Cerrado) — GLM NegBin (≈ seu glm.nb)
# --------------------------------------------------------------------------
sud = dfm[dfm["Bioma"].isin(["Mata Atlântica","Cerrado"])].copy()
# relevels
sud["Bioma"] = sud["Bioma"].cat.remove_unused_categories()
if "Mata Atlântica" in list(sud["Bioma"].cat.categories):
    sud["Bioma"] = sud["Bioma"].cat.reorder_categories(["Mata Atlântica","Cerrado"], ordered=False)

form_nb = "C_aurita_n ~ scale_elev + scale_prec + C(uso_terra_agrupado) + C(Bioma)"
glm_nb = smf.glm(formula=form_nb, data=sud, family=sm.families.NegativeBinomial()).fit()
print("\n--- GLM NegBin (Sudeste) ---")
print(glm_nb.summary())

# --------------------------------------------------------------------------
# GRÁFICOS (não-mapas)
# --------------------------------------------------------------------------

# 1) Contagem total por grupo (barras)
totais = pd.DataFrame({
    "Grupo": ["C. aurita","C. jacchus","C. penicillata","Híbridos"],
    "Contagem": [
        dados_agregados["C_aurita_n"].sum(),
        dados_agregados["C_jacchus_n"].sum(),
        dados_agregados["C_penicillata_n"].sum(),
        dados_agregados["hibridos_n"].sum()
    ]
})
plt.figure(figsize=(6,4))
sns.barplot(data=totais, x="Grupo", y="Contagem")
for i, v in enumerate(totais["Contagem"]):
    plt.text(i, v, f"{int(v)}", ha="center", va="bottom")
plt.title("Contagem Total de Registros por Grupo")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# 2) Boxplot das contagens por célula (somente >0)
long_box = (
    dados_agregados[["id","C_aurita_n","C_jacchus_n","C_penicillata_n","hibridos_n"]]
    .rename(columns={"C_aurita_n":"C. aurita","C_jacchus_n":"C. jacchus",
                     "C_penicillata_n":"C. penicillata","hibridos_n":"Híbridos"})
    .melt(id_vars="id", var_name="Grupo", value_name="Contagem")
)
long_box = long_box[long_box["Contagem"] > 0]

plt.figure(figsize=(7,4))
sns.boxplot(data=long_box, x="Grupo", y="Contagem")
plt.title("Distribuição das Contagens por Célula (apenas >0)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# 3) Urbanos vs Não-Urbanos (barras lado a lado)
urban = (
    dados_agregados
    .assign(Tipo_Area = np.where(dados_agregados["uso_terra"]==24, "Urbano","Não-Urbano"))
    .groupby("Tipo_Area", as_index=False)
    .agg(**{
        "C. aurita": ("C_aurita_n","sum"),
        "C. jacchus": ("C_jacchus_n","sum"),
        "C. penicillata": ("C_penicillata_n","sum"),
        "Híbridos": ("hibridos_n","sum")
    })
    .melt(id_vars="Tipo_Area", var_name="Grupo", value_name="Total_Registros")
)

plt.figure(figsize=(7,4))
sns.barplot(data=urban, x="Grupo", y="Total_Registros", hue="Tipo_Area")
plt.title("Total de Registros em Áreas Urbanas vs. Não-Urbanas")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# 4) Densidade de elevação por grupo (normalizada individualmente)
dens_df = (
    dados_agregados[["elevacao","C_aurita_n","C_jacchus_n","C_penicillata_n"]]
    .rename(columns={"C_aurita_n":"C. aurita","C_jacchus_n":"C. jacchus","C_penicillata_n":"C. penicillata"})
    .melt(id_vars="elevacao", var_name="Grupo", value_name="Contagem")
    .query("Contagem > 0 and elevacao == elevacao")  # remove NA
)

# calcula densidade por grupo e normaliza pico = 1
curvas = []
for g, sub in dens_df.groupby("Grupo"):
    x = sub["elevacao"].values
    if len(x) < 5:
        continue
    kde = stats.gaussian_kde(x)  # parecido com density() do R
    xs = np.linspace(x.min(), x.max(), 512)
    ys = kde(xs)
    ys_norm = ys/ys.max() if ys.max() > 0 else ys
    curvas.append(pd.DataFrame({"x": xs, "dens_norm": ys_norm, "Grupo": g}))
curvas = pd.concat(curvas, ignore_index=True) if curvas else pd.DataFrame()

plt.figure(figsize=(7,4))
for g, sub in curvas.groupby("Grupo"):
    plt.fill_between(sub["x"], sub["dens_norm"], alpha=0.6, label=g)
plt.gca().yaxis.set_major_formatter(lambda v, pos: f"{100*v:.0f}%")
plt.title("Distribuição de Registros por Faixa de Elevação (normalizada por grupo)")
plt.xlabel("Elevação (m)")
plt.ylabel("Frequência relativa")
plt.legend()
plt.tight_layout()
plt.show()

# 5) Gráfico comparando coeficientes dos quatro modelos GLM (com IC 95%)
def coef_table(model, grupo):
    # extrai coeficientes e IC do statsmodels GLM
    params = model.params
    conf = model.conf_int()
    out = pd.DataFrame({
        "term": params.index,
        "estimate": params.values,
        "conf.low": conf[0].values,
        "conf.high": conf[1].values,
        "Grupo": grupo
    })
    # remove intercepto para o gráfico
    out = out[out["term"] != "Intercept"]
    return out

# Modelos GLM negativos binomiais para cada grupo (como no seu bloco comparativo):
m_aurita = smf.glm("C_aurita_n ~ scale_elev + scale_prec + C(uso_terra_agrupado) + C(Bioma)",
                   data=dfm, family=sm.families.NegativeBinomial()).fit()
m_jacchus = smf.glm("C_jacchus_n ~ scale_elev + scale_prec + C(uso_terra_agrupado) + C(Bioma)",
                    data=dfm, family=sm.families.NegativeBinomial()).fit()
m_penic = smf.glm("C_penicillata_n ~ scale_elev + scale_prec + C(uso_terra_agrupado) + C(Bioma)",
                  data=dfm, family=sm.families.NegativeBinomial()).fit()
m_hib = smf.glm("hibridos_n ~ scale_elev + scale_prec + C(uso_terra_agrupado) + C(Bioma)",
                data=dfm, family=sm.families.NegativeBinomial()).fit()

tbl = pd.concat([
    coef_table(m_aurita, "C. aurita (Nativo)"),
    coef_table(m_jacchus, "C. jacchus (Invasor)"),
    coef_table(m_penic, "C. penicillata (Invasor)"),
    coef_table(m_hib, "Híbridos")
], ignore_index=True)

# "Tradução" de nomes para legenda legível
map_terms = {
    "scale_elev": "Elevação",
    "scale_prec": "Precipitação Média",
}
tbl["termo_limpo"] = tbl["term"].replace(map_terms, regex=False)
tbl["termo_limpo"] = tbl["termo_limpo"].str.replace(r'^C\(uso_terra_agrupado\)\[T\.Area_Urbanizada\]$', "Uso: Área Urbanizada", regex=True)
tbl["termo_limpo"] = tbl["termo_limpo"].str.replace(r'^C\(uso_terra_agrupado\)\[T\.Uso_Antropico_Rural\]$', "Uso: Antrópico Rural", regex=True)
tbl["termo_limpo"] = tbl["termo_limpo"].str.replace(r'^C\(uso_terra_agrupado\)\[T\.Outros_e_Abertos\]$', "Uso: Áreas Abertas", regex=True)
tbl["termo_limpo"] = tbl["termo_limpo"].str.replace(r'^C\(Bioma\)\[T\.Cerrado\]$', "Bioma: Cerrado", regex=True)

# ordenação dos painéis
cat_order = ["C. aurita (Nativo)","C. jacchus (Invasor)","C. penicillata (Invasor)","Híbridos"]
tbl["Grupo"] = pd.Categorical(tbl["Grupo"], categories=cat_order, ordered=True)

# plota (facetas)
g = sns.FacetGrid(tbl, col="Grupo", col_wrap=2, sharex=False, height=4)
def facet_plot(data, **kwargs):
    y = data["termo_limpo"]
    x = data["estimate"]
    xmin = data["conf.low"]
    xmax = data["conf.high"]
    idx = np.arange(len(data))
    plt.hlines(idx, xmin, xmax)
    plt.plot(x, idx, 'o')
    plt.yticks(idx, y)
    plt.axvline(0, ls="--", c="red")
g.map_dataframe(facet_plot)
g.set_xlabels("Estimativa (IC95%)")
g.set_ylabels("")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Efeito das Variáveis Ambientais na Ocorrência de Saguis")
plt.show()

print("\nFIM ✅")


ModuleNotFoundError: No module named 'statsmodels'